# Analyze Study (Regression)

This notebook walks through the post-hoc analysis of a completed Octopus regression study. It covers study validation, performance evaluation, feature selection summaries, prediction diagnostics, and feature importance analysis.

Each section can be run independently once the study is loaded.

**Prerequisite:** Run `basic_regression.py` first to generate the study data:

```bash
python examples/basic_regression.py
```

In [ ]:
from octopus.analysis import (
    feature_count_plot,
    feature_frequency_plot,
    fi_plot,
    get_details,
    load_study_info,
    performance,
    performance_plot,
    performance_table,
    prediction_plot,
    residual_plot,
    selected_features,
    workflow_graph,
)

## Load Study

Provide the path to your study directory. If you pass a name prefix (e.g. `"../studies/my_study"`), `load_study_info` automatically finds the latest timestamped run matching that prefix. The study directory is validated on load -- missing outersplits or workflow task directories raise an error immediately.


In [ ]:
study_info = load_study_info("../studies/basic_regression")

## Study Details

`get_details` returns a summary of the study configuration (ML type, target metric, number of folds, and available outer split IDs).

`workflow_graph` displays the workflow tasks and their dependencies as a tree. Each node shows the task ID, module type, and description. Branches indicate that multiple tasks depend on the same parent.


In [ ]:
details = get_details(study_info)
details

In [ ]:
workflow = workflow_graph(study_info)
print(workflow)

## Performance

`performance` returns metric scores per outer split for all prediction tasks in the workflow. Feature selection tasks are skipped automatically. By default the `dev` partition is shown -- use this partition for model selection and hyperparameter decisions to avoid data leakage from the held-out test set. Only use `partition="test"` for final, unbiased evaluation after all decisions have been made. Use `metric` to filter to a single metric, or `task` to select a specific workflow task.

**How to read the plot:**

- Each bar represents the metric score for one outer split. Consistent bar heights across splits indicate stable model performance.
- Large variation between splits may signal that the model is sensitive to the particular train/test partition, which can happen with small datasets or high-variance targets.
- When multiple tasks are shown, comparing bars across tasks reveals whether later workflow steps (e.g. after feature selection) improve or degrade performance.
- For error metrics like MAE, MSE, and RMSE, lower values are better. For R2, higher values (closer to 1.0) are better.


In [ ]:
perf = performance(study_info)
fig = performance_plot(perf)
fig.show()

### Performance Table

`performance_table` shows the train and dev scores for a single task and metric side by side, with one row per outer split and a Mean row. Comparing train and dev columns reveals whether the model is overfitting (large gap) or underfitting (both scores poor).


In [ ]:
performance_table(study_info, task=0, metric="MAE")

## Selected Features

`selected_features` reads `selected_features.json` from each outersplit and task directory and returns two tables:

- **feature_table**: number of selected features per outersplit and task (plus a Mean row).
- **frequency_table**: how often each feature was selected across outersplits, sorted descending.

**How to read the plots:**

- The **feature count plot** shows how many features were selected in each outer split (and task). A consistent count across splits suggests a stable feature selection. A large drop between workflow tasks confirms that feature selection steps are reducing dimensionality as intended.
- The **feature frequency plot** shows which features are most consistently selected across outer splits. Features selected in all splits are robust and likely genuinely informative. Features selected in only one or two splits may be noise or split-specific artefacts.


In [ ]:
feature_table, frequency_table = selected_features(study_info)
fig = feature_count_plot(feature_table)
fig.show()

In [ ]:
fig = feature_frequency_plot(frequency_table)
fig.show()

## Test Set Evaluation

The sections above use the **dev** partition -- these results should guide model selection, hyperparameter tuning, and feature selection decisions. Looking at test scores during these steps introduces data leakage and inflates reported performance.

The **test** partition below provides an unbiased estimate of final model performance. Only look at test results **after all modelling decisions have been made**. If test results lead you to change the model or features, the test set is no longer independent and the reported scores lose their validity.

`OctoTestEvaluator` re-computes metrics on the held-out test data per outersplit.


In [ ]:
from octopus.analysis import OctoTestEvaluator

task_predictor_test = OctoTestEvaluator(study=study_info, task_id=0, result_type="best")

## Prediction vs Ground Truth

`prediction_plot` shows a scatter plot of predicted values against the true target values for each outer split. The dashed diagonal line represents perfect predictions.

**How to read the plot:**

- Points close to the diagonal indicate accurate predictions. The tighter the cloud around the line, the better the model.
- Systematic deviations from the diagonal reveal bias. For example, if points curve below the line at high target values, the model underestimates large values.
- Wide scatter indicates high prediction variance. If scatter is consistent across splits, it reflects a fundamental limitation; if it varies, the model may be sensitive to the train/test partition.


In [ ]:
fig = prediction_plot(task_predictor_test)
fig.show()

## Residual Plot

`residual_plot` shows residuals (prediction minus ground truth) plotted against the predicted values for each outer split. The dashed horizontal line at zero marks perfect predictions.

**How to read the plot:**

- Ideally, residuals should be randomly scattered around zero with no visible pattern. This indicates that the model's errors are unstructured.
- A funnel shape (residuals growing with predicted value) suggests heteroscedasticity -- the model is less reliable for certain value ranges.
- A curved pattern indicates the model is systematically over- or under-predicting in certain regions, which may point to a non-linear relationship the model is not capturing.
- Outliers far from zero deserve investigation -- they may be data errors or genuinely hard-to-predict samples.


In [ ]:
fig = residual_plot(task_predictor_test)
fig.show()

## Feature Importance

Feature importance measures how much each feature contributes to the model's predictions. `calculate_fi` computes importance on the held-out test data using the final fitted models. Each outer-split model is evaluated independently on its own test fold, then results are aggregated into an ensemble summary.

Two methods are available:

- **Permutation importance** (`fi_type="permutation"`): Randomly shuffles each feature and measures the drop in model performance. A large drop means the feature is important. Use `"group_permutation"` to also compute importance at the feature-group level. Fast and model-agnostic.
- **SHAP** (`fi_type="shap"`): Computes Shapley values that attribute the prediction to individual features. More fine-grained than permutation but slower. Available `shap_type` options: `"kernel"` (default, model-agnostic), `"permutation"`, `"exact"` (slowest, most accurate).

**How to read the plot:**

- Features are sorted by importance. The top features contribute most to the model's predictions.
- Error bars (when available) show confidence intervals across outer splits. Narrow bars indicate consistent importance, wide bars suggest the feature's contribution varies by split.
- Features with importance near zero or negative can often be removed without hurting performance.


### Permutation Feature Importance

`n_repeats` controls how many times each feature is shuffled. Higher values give more stable importance estimates and tighter confidence intervals. For this example we use `n_repeats=3` to keep runtime short. For real analyses, `n_repeats=10` (default) or higher is recommended.


In [ ]:
fi_table_perm = task_predictor_test.calculate_fi(fi_type="permutation", n_repeats=3)
fig = fi_plot(fi_table_perm)
fig.show()

### SHAP Feature Importance


In [ ]:
fi_table_shap = task_predictor_test.calculate_fi(fi_type="shap", shap_type="kernel")
fig = fi_plot(fi_table_shap)
fig.show()